[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/03_Choosing_Your_Model.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/03_Choosing_Your_Model.ipynb)

# Reading Data to Choose a Model — A Step-by-Step Checklist

You receive a new dataset. What model should you use?

This notebook walks through every step a practitioner uses to answer that question.
Each step uses statistics and visualisation — **before** training a single model.

---

**The checklist**

1. What type is the target? → regression or classification
2. What does the target distribution look like?
3. Which features correlate with the target?
4. Do the relationships look linear or do they show curves and jumps?
5. Fit a simple baseline model and check its errors
6. Fit a decision tree — does it do better, and why?

We apply the full checklist to the **Titanic dataset** — familiar from earlier notebooks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

df_raw = sns.load_dataset('titanic')
print(f"Titanic: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw[['survived','pclass','sex','age','fare']].head(6)

## Step 1 — What Type Is the Target?

Count the unique values of the target column.

- **Two values (0 / 1)** → binary classification → Logistic Regression or Decision Tree
- **A few categories** → multi-class classification
- **Continuous numbers** → regression → Linear Regression or Decision Tree Regressor

The type of target determines which family of models you can use.

In [ ]:
counts = df_raw['survived'].value_counts()
print("Unique values in 'survived':", sorted(df_raw['survived'].unique()))
print(counts.to_string())
print()
print("Only 0 and 1 — this is a binary classification problem.")
print(f"Base rate: {counts[1]/counts.sum():.0%} survived — predicting 'died' for everyone gives {counts[0]/counts.sum():.0%} accuracy for free")

fig, ax = plt.subplots(figsize=(5, 3))
counts.plot(kind='bar', color=['#e57373', '#4CAF50'], ax=ax, rot=0)
ax.set_xticklabels(['Died (0)', 'Survived (1)'])
ax.set_ylabel('Count')
ax.set_title('Target distribution: survived', fontsize=11)
plt.tight_layout()
plt.show()

## Step 2 — Which Features Correlate With Survival?

For a 0/1 target, Pearson correlation still works — it measures how much each feature
moves in the direction of survival.

**High absolute correlation** → that feature separates survivors from non-survivors.
**Near zero** → that feature has little linear predictive power.

This step tells you which features are worth including in a model.

In [ ]:
df = df_raw[['survived','pclass','age','fare','sex']].copy()
df['sex_num'] = (df['sex'] == 'male').astype(int)
df = df.dropna()

features = ['pclass', 'age', 'fare', 'sex_num']
corr = df[features + ['survived']].corr()['survived'].drop('survived').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bar_colors = ['#e57373' if v < 0 else '#4CAF50' for v in corr]
corr.plot(kind='barh', color=bar_colors, ax=axes[0])
axes[0].axvline(0, color='black', lw=1)
axes[0].set_title('Correlation with survival\nRed = associated with dying   Green = associated with surviving', fontsize=10)
axes[0].set_xlabel('Pearson r')

surv_by_class = df_raw.groupby('pclass')['survived'].mean()
surv_by_class.plot(kind='bar', color='steelblue', ax=axes[1], rot=0)
axes[1].set_xticklabels(['1st class', '2nd class', '3rd class'])
axes[1].set_ylabel('Survival rate')
axes[1].set_title('Survival rate by passenger class\nStrong pattern — class matters a lot', fontsize=10)

plt.tight_layout()
plt.show()

print("pclass:   negative — higher class number (3rd) = lower survival")
print("sex_num:  negative — male = 1, lower survival (women and children first)")
print("fare:     positive — higher fare = wealthier = better cabin access = higher survival")
print("age:      weak — survival rate does not change proportionally with age")

## Step 3 — Are the Relationships Linear or Not?

For a classification target, we ask:
**Does the survival rate change smoothly and proportionally as each feature increases?**

- If yes → logistic regression can capture it with a smooth S-curve
- If the rate jumps at a specific threshold, or depends on two features together → need a tree

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survival rate vs age (binned) — check for threshold effects
df_raw['age_bin'] = pd.cut(df_raw['age'], bins=[0, 10, 20, 30, 40, 50, 60, 80])
age_surv = df_raw.groupby('age_bin', observed=True)['survived'].mean()
age_surv.plot(kind='bar', color='steelblue', ax=axes[0], rot=45)
axes[0].set_title('Survival rate by age group\nChildren under 10 had much higher survival\n→ threshold effect, not a smooth line', fontsize=9)
axes[0].set_ylabel('Survival rate')
axes[0].set_xlabel('')

# Survival rate vs fare (binned) — check for linearity
df_raw['fare_bin'] = pd.qcut(df_raw['fare'], q=5, duplicates='drop')
fare_surv = df_raw.groupby('fare_bin', observed=True)['survived'].mean()
fare_surv.plot(kind='bar', color='#26A69A', ax=axes[1], rot=45)
axes[1].set_title('Survival rate by fare group\nRoughly increasing — more linear\n→ logistic may handle this', fontsize=9)
axes[1].set_ylabel('Survival rate')
axes[1].set_xlabel('')

# Interaction: sex AND class together
pivot = df_raw.groupby(['sex', 'pclass'])['survived'].mean().unstack()
pivot.plot(kind='bar', ax=axes[2], rot=0, color=['#5C6BC0', '#FFA726', '#e57373'])
axes[2].set_title('Survival by sex AND class together\nFemale+1st class = 97%   Male+3rd class = 14%\n→ strong interaction between two features', fontsize=9)
axes[2].set_ylabel('Survival rate')
axes[2].legend(title='Class', labels=['1st', '2nd', '3rd'])

plt.tight_layout()
plt.show()

print("Age:       non-linear — threshold effect for children under 10")
print("Fare:      roughly monotonic — logistic regression may work")
print("Sex×Class: strong interaction — one feature alone does not tell the full story")
print()
print("Conclusion: the data has thresholds and interactions — a decision tree should outperform logistic regression")

## Step 4 — Train Both Models and Compare

Now we train the two models the data analysis suggested:

- **Logistic Regression** — assumes a smooth linear decision boundary
- **Decision Tree** — learns IF-THEN rules that capture thresholds and interactions

We check how well each one actually predicts survival.

In [ ]:
X = df[features].values
y = df['survived'].values

X_scaled = StandardScaler().fit_transform(X)

lr_model   = LogisticRegression(max_iter=1000).fit(X_scaled, y)
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X, y)

lr_acc   = accuracy_score(y, lr_model.predict(X_scaled))
tree_acc = accuracy_score(y, tree_model.predict(X))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    y, lr_model.predict(X_scaled), ax=axes[0],
    display_labels=['Died', 'Survived'], colorbar=False
)
axes[0].set_title(f'Logistic Regression\nAccuracy = {lr_acc:.1%}', fontsize=12)

ConfusionMatrixDisplay.from_predictions(
    y, tree_model.predict(X), ax=axes[1],
    display_labels=['Died', 'Survived'], colorbar=False
)
axes[1].set_title(f'Decision Tree (depth=4)\nAccuracy = {tree_acc:.1%}', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Logistic Regression: {lr_acc:.1%}")
print(f"Decision Tree:       {tree_acc:.1%}")
print()
print("The tree is better because the data has:")
print("  - A threshold effect (children under 10 survive more)")
print("  - An interaction (sex AND class together predict survival better than either alone)")
print("Logistic regression draws a smooth straight boundary — it cannot capture sharp rules.")

## Step 5 — Where Did Logistic Regression Fail?

Logistic regression makes one smooth prediction: "each additional unit of X increases survival probability by Y%."

But the data violates this assumption:

**Age**: survival probability is NOT proportional to age.
Children under 10 had unusually high survival. For adults, age barely mattered.
A straight line cannot have a spike at age < 10.

**Sex + Class interaction**: a female in 1st class had 97% survival.
A male in 3rd class had 14%.
Logistic regression would need manually engineered features to capture this.
The decision tree finds it automatically with two splits: "sex = female?" then "class = 1 or 2?"

This is why the **residual / error analysis** matters:
inspect *which* observations the linear model gets wrong — they reveal the structure it missed.

## The Full Checklist

Use these questions every time you see a new dataset:

| Question to ask | If YES | If NO / unsure |
|---|---|---|
| Target has 2 values (0/1)? | Classification | Regression (continuous target) |
| Features correlate linearly with target? | Try Logistic/Linear first | Scatter/bar plots may reveal curves |
| Survival/outcome rate is smooth across feature values? | Linear model may work | Decision tree needed |
| Outcome jumps at a threshold? | Decision tree | Linear may still work |
| Two features interact (one only matters when another is high)? | Decision tree / Random Forest | Linear may still work |
| You need a human-readable rule? | Decision tree | Any model |

### Step-by-step summary

```
1. Look at target distribution   → what problem type?
2. Compute correlations          → which features matter?
3. Bar/scatter plots             → are relationships straight or do they jump?
4. Fit linear/logistic baseline  → check errors and confusion matrix
5. If errors show patterns       → try a Decision Tree
6. If tree wins                  → consider Random Forest for better generalisation
```

## Key Takeaways

1. **Statistics before models** — correlation and plots tell you what the model will learn before it trains.
2. **Linear/logistic** works when the relationship is smooth and proportional.
3. **Threshold effects** (outcome jumps at a value) and **interactions** (two features together) break linear models.
4. **Decision trees** learn IF-THEN rules that naturally capture thresholds and interactions.
5. **The error pattern tells you what you missed** — look at which observations the model got wrong.
6. **Random Forest is the safe default** — it builds many trees to reduce overfitting and handles both linear and non-linear patterns.